In [1]:
chr(0)

'\x00'

In [18]:
print(list('c'.__repr__()))

["'", 'c', "'"]


In [20]:
chr(0)

'\x00'

In [21]:
print(chr(0))

 


In [ ]:
s = "this is a test" + chr(0) + "string"
s

'this is a test\x00string'

In [38]:
test_string = "hello! こんにちは!"
utf8_encoded = test_string.encode('utf-8')
print(type(utf8_encoded))
print(list(utf8_encoded))

<class 'bytes'>
[104, 101, 108, 108, 111, 33, 32, 227, 129, 147, 227, 130, 147, 227, 129, 171, 227, 129, 161, 227, 129, 175, 33]


In [40]:
print(utf8_encoded.decode('utf-8'))

hello! こんにちは!


#### Problem (unicode 2) Unicode Encodings (3 points)

In [ ]:
# (a) Because UTF-16 and UTF-32 are fixed length encodings which waste a lot of space 
# if we encode mostly ASCI.

In [51]:
# (b)
def decode_wrong(bytestring: bytes):
    return "".join([bytes([b]).decode('utf-8') for b in bytestring])

In [73]:
two_bytes = "é".encode('utf-8')
print(list(two_bytes))
try:
    decode_wrong(two_bytes)
except:
    print("decoding failed!")
    
# well, as long as a character corresponds to one byte in the encoded representation,
# the above function works well. However, UTF-8 is variable-length.

[195, 169]
decoding failed!


In [1]:
import torch

torch.Size([3, 3, 4])

In [134]:
x

tensor([[[ 1,  2,  3,  4],
         [ 5,  6,  7,  8],
         [ 9, 10, 11, 12]],

        [[13, 14, 15, 16],
         [17, 18, 19, 20],
         [21, 22, 23, 24]],

        [[25, 26, 27, 28],
         [29, 30, 31, 32],
         [33, 34, 35, 36]]])

In [152]:
torch.sum(x, dim=(2,), keepdim=True).shape

torch.Size([3, 3, 1])

In [35]:
torch.max(x, dim=2).values  

tensor([[ 4,  8, 12],
        [16, 20, 24],
        [28, 32, 36]])

In [171]:
x = torch.arange(1, 1+3*3*4).reshape(3,3,4)
x.shape
x = x - x.max(dim=1, keepdim=True).values

In [181]:
v = x.exp() / x.exp().sum(dim=1, keepdim=True)
v

tensor([[[3.2932e-04, 3.2932e-04, 3.2932e-04, 3.2932e-04],
         [1.7980e-02, 1.7980e-02, 1.7980e-02, 1.7980e-02],
         [9.8169e-01, 9.8169e-01, 9.8169e-01, 9.8169e-01]],

        [[3.2932e-04, 3.2932e-04, 3.2932e-04, 3.2932e-04],
         [1.7980e-02, 1.7980e-02, 1.7980e-02, 1.7980e-02],
         [9.8169e-01, 9.8169e-01, 9.8169e-01, 9.8169e-01]],

        [[3.2932e-04, 3.2932e-04, 3.2932e-04, 3.2932e-04],
         [1.7980e-02, 1.7980e-02, 1.7980e-02, 1.7980e-02],
         [9.8169e-01, 9.8169e-01, 9.8169e-01, 9.8169e-01]]])

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from einops import einsum

from cs336_basics.model.softmax import Softmax

def forward(self, Q: torch.Tensor, K: torch.Tensor, V: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        softmax = F.softmax()
        # A = Q @ K.T
        d_k =torch.tensor(Q.shape[-1], dtype=torch.float32)   
        A = (torch.sqrt(d_k) ** -1) * einsum(Q, K, "batch_size ... i j, batch_size ... k j -> batch_size ... i k") 
        
        return einsum(A, V, "batch_size ... n m, batch_size ... m d_v -> batch_size ... n d_v")

In [191]:
n = 8
m = 5
d_k = 3
d_v = 2

In [211]:
Q = torch.randn(n, d_k)
K = torch.randn(m, d_k)
V = torch.randn(m, d_v)
d = torch.sqrt(torch.tensor(Q.shape[-1], dtype=torch.float32))

In [212]:
result = F.softmax(Q@K.T / d, dim=-1) @ V

In [213]:
result

tensor([[ 0.2924,  0.4520],
        [-0.1699,  0.4626],
        [-0.4399,  0.3288],
        [ 0.1239,  0.2617],
        [ 0.3508,  0.2232],
        [-0.0467,  0.1388],
        [-0.5916,  0.1915],
        [-0.3976,  0.1615]])

In [214]:
d = torch.sqrt(torch.tensor(Q.shape[-1], dtype=torch.float32))

In [217]:
scores = (d**-1) * einsum(Q, K, "... i j, ... k j -> ... i k")
weights = F.softmax(scores, dim=-1)
result2 = einsum(weights, V, "... n m, ... m d_v -> n d_v")

In [224]:
mask = torch.randint(low=0, high=2, size=(n,m))

In [225]:
result[mask] = -torch.inf

In [ ]:
d = torch.sqrt(torch.tensor(Q.shape[-1], dtype=torch.float32))
scores = (d**-1) * einsum(Q, K, "... i j, ... k j -> ... i k")

scores += scores[] -torch.inf
weights = F.softmax(scores, dim=-1)
result = einsum(weights, V, "... n m, ... m d_v -> ... n d_v")

In [237]:
x = torch.arange(1, 11)

In [238]:
x

tensor([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10])

In [239]:
from einops import rearrange

In [249]:
rearrange(x, '(a b) -> a b', b = 2)

tensor([[ 1,  2],
        [ 3,  4],
        [ 5,  6],
        [ 7,  8],
        [ 9, 10]])

In [250]:
mask = torch.ones((5,5))

In [251]:
mask

tensor([[1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1.]])

In [253]:
torch.triu(mask).T

tensor([[1., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0.],
        [1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1.]])